<a href="https://colab.research.google.com/github/marfenix/Apprenticeship/blob/main/AIDI%20NLP%20Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Production Retail Supply Chain NLP Pipeline (FinBERT)

### Pipeline Overview
1. **Environment Setup & Directory Architecture**: Automated dependency configuration and directory management (`./data/raw`, `./data/processed`).
2. **Synthetic Data Generation (Privacy-by-Design)**: Programmatic generation of realistic supplier emails and market news covering four key commodities (`wheat`, `sugar`, `cereal`, `packaging`) across October 2023, with deliberate edge-case anomalies for quality testing.
3. **Ingestion, Unit Testing & Data Quality Verification**: Schema validation with explicit assertions, whitespace/null cleaning, and ingestion audit logging.
4. **Commodity Keyword Tagging**: Deterministic multi-keyword tagging with unit test assertions.
5. **FinBERT Sentiment Inference**: Batched inference with `ProsusAI/finbert` via Hugging Face Transformers pipeline, GPU acceleration check, label standardization, and score rounding.
6. **Business Logic — 2-Day Rolling Risk Escalation**: Chronological daily net risk computation per commodity with a 2-day consecutive risk escalation matrix (`High`, `Watch`, `Normal`).
7. **Model Evaluation & Error Analysis**: Binary performance metrics (Accuracy, Precision, Recall, F1), scikit-learn classification report, confusion matrix, confidence distribution audits, and misclassification error analysis.
8. **Export & Summary Audit**: Power BI-ready curated export to `./data/processed/sentiment_output.csv` with executive execution logs.

In [ ]:
# =============================================================================
# Cell 1: Environment Setup & Directory Creation
# =============================================================================
# Install core dependencies silently for Colab compatibility
!pip install -q transformers torch scikit-learn pandas

import os
import csv
import warnings
import numpy as np
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# Programmatically create production folder structure
RAW_DATA_DIR = "./data/raw"
PROCESSED_DATA_DIR = "./data/processed"

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"[SETUP COMPLETE] Raw data directory: {RAW_DATA_DIR}")
print(f"[SETUP COMPLETE] Processed data directory: {PROCESSED_DATA_DIR}")
print(f"[COMPUTE ACCELERATOR] CUDA Available: {torch.cuda.is_available()}")

[SETUP COMPLETE] Raw data directory: ./data/raw
[SETUP COMPLETE] Processed data directory: ./data/processed
[COMPUTE ACCELERATOR] CUDA Available: False


In [ ]:
# =============================================================================
# Cell 2: Synthetic Data Generation (Privacy-by-Design)
# =============================================================================
# Synthetic supplier emails (25-30 records, October 2023)
# Features: wheat, sugar, cereal, packaging with positive, negative, and neutral conditions
# Deliberately contains 2 invalid rows (null / whitespace text) to test data validation
supplier_emails_data = [
    (1, "2023-10-01", "Invoice #402 for wheat delivery processed successfully.", "supplier_email", "NEUTRAL"),
    (2, "2023-10-02", "Bumper harvest for wheat means early delivery next week at contracted rate!", "supplier_email", "POSITIVE"),
    (3, "2023-10-03", "Severe drought hitting wheat farms in Midwest, expect potential dispatch delays.", "supplier_email", "NEGATIVE"),
    (4, "2023-10-03", "Routine delivery scheduling for cereal shipments to regional distribution center.", "supplier_email", "NEUTRAL"),
    (5, "2023-10-04", "Severe drought continues across agricultural belt, wheat shipments canceled.", "supplier_email", "NEGATIVE"),
    (6, "2023-10-05", "Sugar refinery operations resumed, all machinery back to normal capacity.", "supplier_email", "POSITIVE"),
    (7, "2023-10-05", "Energy grid failures at sugar refineries causing massive order backlog.", "supplier_email", "NEGATIVE"),
    (8, "2023-10-06", "Packaging shortages mitigated, primary corrugated supplier onboarded successfully.", "supplier_email", "POSITIVE"),
    (9, "2023-10-07", "Rising pulp packaging costs severely affecting operating margins this quarter.", "supplier_email", "NEGATIVE"),
    (10, "2023-10-08", "Cereal bulk packaging dispatch remains strictly on schedule.", "supplier_email", "NEUTRAL"),
    (11, "2023-10-08", "Annual wheat contract negotiations resolved favorably with guaranteed volumes.", "supplier_email", "POSITIVE"),
    (12, "2023-10-09", None, "supplier_email", "NEUTRAL"),  # Edge case: None text
    (13, "2023-10-10", "Port strikes affecting sugar container shipments and export terminals.", "supplier_email", "NEGATIVE"),
    (14, "2023-10-10", "   ", "supplier_email", "NEUTRAL"),  # Edge case: Whitespace only text
    (15, "2023-10-11", "Scheduled delivery of packaging materials confirmed for warehouse bay 4.", "supplier_email", "NEUTRAL"),
    (16, "2023-10-12", "Port strikes continue unabated, sugar raw deliveries delayed indefinitely.", "supplier_email", "NEGATIVE"),
    (17, "2023-10-13", "New organic cereal recipe approved for commercial production and distribution.", "supplier_email", "POSITIVE"),
    (18, "2023-10-14", "Wheat harvest yield significantly better than initial pessimistic forecasts.", "supplier_email", "POSITIVE"),
    (19, "2023-10-15", "Secondary packaging cardboard costs stabilizing after supplier renegotiation.", "supplier_email", "POSITIVE"),
    (20, "2023-10-16", "Routine quality laboratory inspection for refined sugar passed with zero defects.", "supplier_email", "NEUTRAL"),
    (21, "2023-10-17", "Cereal raw ingredient shortage causing a 48-hour plant stoppage.", "supplier_email", "NEGATIVE"),
    (22, "2023-10-18", "Consignment of durum wheat delivered on time.", "supplier_email", "NEUTRAL"),
    (23, "2023-10-19", "Sugar multi-year supply contract renewed with attractive volume discount.", "supplier_email", "POSITIVE"),
    (24, "2023-10-20", "Packaging seal defects discovered in batch 89, recall instituted.", "supplier_email", "NEGATIVE"),
    (25, "2023-10-21", "National retail marketing promotion for cereal brand commenced successfully.", "supplier_email", "POSITIVE"),
    (26, "2023-10-22", "Standard weekly confirmation of packaging pallets received.", "supplier_email", "NEUTRAL")
]

# Synthetic market news articles (25-30 records, October 2023)
# Deliberately contains 2 invalid rows (empty string / NaN text)
news_articles_data = [
    (101, "2023-10-01", "Global wheat market commodity prices drop sharply due to record bumper harvests.", "news_article", "POSITIVE"),
    (102, "2023-10-02", "Energy grid failures hit major regional sugar producers, halting refining.", "news_article", "NEGATIVE"),
    (103, "2023-10-03", "Breakfast cereal market demand remains stable this quarter, analysts report.", "news_article", "NEUTRAL"),
    (104, "2023-10-04", "Breakthrough innovations in biodegradable packaging reduce manufacturer costs by 15%.", "news_article", "POSITIVE"),
    (105, "2023-10-05", "Severe drought conditions in key wheat-producing regions threaten winter supply.", "news_article", "NEGATIVE"),
    (106, "2023-10-06", "Severe drought worsens across grain belt, wheat harvest yields plummet to historical lows.", "news_article", "NEGATIVE"),
    (107, "2023-10-07", "Sugar refineries invest in on-site solar energy to insulate against grid outages.", "news_article", "POSITIVE"),
    (108, "2023-10-08", "Surging pulp packaging costs place severe strain on food manufacturing supply chains.", "news_article", "NEGATIVE"),
    (109, "2023-10-09", "", "news_article", "NEUTRAL"),  # Edge case: Empty string
    (110, "2023-10-10", "Top cereal manufacturers announce consumer price cuts amid grain surplus.", "news_article", "POSITIVE"),
    (111, "2023-10-11", "Nationwide dock strikes threaten to paralyze key sugar import and distribution corridors.", "news_article", "NEGATIVE"),
    (112, "2023-10-12", "Prolonged port strikes shut down maritime freight terminals, halting international sugar trade.", "news_article", "NEGATIVE"),
    (113, "2023-10-13", "Leading consumer brands transition toward lightweight sustainable packaging solutions.", "news_article", "POSITIVE"),
    (114, "2023-10-14", None, "news_article", "NEUTRAL"),  # Edge case: Null value
    (115, "2023-10-15", "Chicago Board of Trade reports wheat futures stabilize following week of high volatility.", "news_article", "NEUTRAL"),
    (116, "2023-10-16", "Ready-to-eat cereal sales surge in retail grocery channels across metropolitan zones.", "news_article", "POSITIVE"),
    (117, "2023-10-17", "Sugar spot prices surge 12% following catastrophic frost in cane growing regions.", "news_article", "NEGATIVE"),
    (118, "2023-10-18", "Strict environmental packaging compliance standards introduce unexpected supply chain overhead.", "news_article", "NEGATIVE"),
    (119, "2023-10-19", "Agricultural authorities confirm crop disease outbreak in grain wheat has been contained.", "news_article", "POSITIVE"),
    (120, "2023-10-20", "Logistics bottlenecks for bulk cereal distribution clear as rail capacity expands.", "news_article", "NEUTRAL"),
    (121, "2023-10-21", "Agribusiness conglomerate announces record quarterly earnings driven by robust sugar sales.", "news_article", "POSITIVE"),
    (122, "2023-10-22", "Corrugated packaging supply deficit projected to escalate throughout next month.", "news_article", "NEGATIVE"),
    (123, "2023-10-23", "Wheat export volumes hit record high as transport infrastructure normalizes.", "news_article", "POSITIVE"),
    (124, "2023-10-24", "Major automated cereal manufacturing facility halted by critical mechanical breakdown.", "news_article", "NEGATIVE"),
    (125, "2023-10-25", "Sugar commodity pricing index records neutral trading day with balanced global stockpiles.", "news_article", "NEUTRAL"),
    (126, "2023-10-26", "Wheat production outlook upgraded for the coming agricultural season.", "news_article", "POSITIVE")
]

def export_raw_csv(records, filepath):
    with open(filepath, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "date", "text", "source_type", "true_label"])
        writer.writerows(records)

supplier_csv_path = os.path.join(RAW_DATA_DIR, "supplier_emails.csv")
news_csv_path = os.path.join(RAW_DATA_DIR, "news_articles.csv")

export_raw_csv(supplier_emails_data, supplier_csv_path)
export_raw_csv(news_articles_data, news_csv_path)

print(f"[DATA GENERATION] Saved {len(supplier_emails_data)} synthetic rows -> {supplier_csv_path}")
print(f"[DATA GENERATION] Saved {len(news_articles_data)} synthetic rows -> {news_csv_path}")

[DATA GENERATION] Saved 26 synthetic rows -> ./data/raw/supplier_emails.csv
[DATA GENERATION] Saved 26 synthetic rows -> ./data/raw/news_articles.csv


In [ ]:
# =============================================================================
# Cell 3: Ingestion, Unit Testing & Data Quality Checks
# =============================================================================
# Ingest raw CSV data
df_emails_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, "supplier_emails.csv"))
df_news_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, "news_articles.csv"))

# -----------------------------------------------------------------------------
# Unit Tests (Assertions)
# -----------------------------------------------------------------------------
assert not df_emails_raw.empty, "[ASSERTION FAILED] supplier_emails.csv is empty"
assert not df_news_raw.empty, "[ASSERTION FAILED] news_articles.csv is empty"

REQUIRED_COLUMNS = ["id", "date", "text", "source_type", "true_label"]
assert all(col in df_emails_raw.columns for col in REQUIRED_COLUMNS), \
    f"[ASSERTION FAILED] Missing columns in supplier_emails: {set(REQUIRED_COLUMNS) - set(df_emails_raw.columns)}"
assert all(col in df_news_raw.columns for col in REQUIRED_COLUMNS), \
    f"[ASSERTION FAILED] Missing columns in news_articles: {set(REQUIRED_COLUMNS) - set(df_news_raw.columns)}"

print("[UNIT TEST PASSED] Raw DataFrames verified non-empty with all required schema columns.")

# -----------------------------------------------------------------------------
# Data Quality Check & Cleaning Implementation
# -----------------------------------------------------------------------------
def clean_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, int, int]:
    """
    Identifies and purges null, NaN, or whitespace-only records in the 'text' column.
    Returns (cleaned_df, initial_row_count, dropped_row_count).
    """
    initial_count = len(df)

    # Identify null or whitespace strings
    is_invalid = df["text"].isna() | (df["text"].astype(str).str.strip() == "")
    invalid_count = int(is_invalid.sum())

    df_clean = df[~is_invalid].copy()
    df_clean["text"] = df_clean["text"].astype(str).str.strip()

    return df_clean, initial_count, invalid_count

df_emails_clean, emails_init, emails_dropped = clean_dataset(df_emails_raw)
df_news_clean, news_init, news_dropped = clean_dataset(df_news_raw)

# Audit Logging Report
print("\n" + "="*60)
print("DATA CLEANING & QUALITY AUDIT REPORT")
print("="*60)
print(f"[Supplier Emails] Ingested: {emails_init:2d} | Dropped: {emails_dropped:2d} | Cleaned: {len(df_emails_clean):2d}")
print(f"[News Articles]   Ingested: {news_init:2d} | Dropped: {news_dropped:2d} | Cleaned: {len(df_news_clean):2d}")
print(f"[Total Purged]    {emails_dropped + news_dropped} invalid rows dropped from raw feeds.")
print("="*60)

[UNIT TEST PASSED] Raw DataFrames verified non-empty with all required schema columns.

DATA CLEANING & QUALITY AUDIT REPORT
[Supplier Emails] Ingested: 26 | Dropped:  2 | Cleaned: 24
[News Articles]   Ingested: 26 | Dropped:  2 | Cleaned: 24
[Total Purged]    4 invalid rows dropped from raw feeds.


In [ ]:
# =============================================================================
# Cell 4: Commodity Keyword Tagging & Tagging Unit Test
# =============================================================================
COMMODITY_KEYWORDS = ["wheat", "sugar", "cereal", "packaging"]

def tag_commodities(text: str) -> str:
    """
    Inspects lowercased text for occurrences of target commodities.
    Returns comma-separated matching commodities, or 'other' if none match.
    """
    if not isinstance(text, str) or not text.strip():
        return "other"
    text_lower = text.lower()
    matches = [kw for kw in COMMODITY_KEYWORDS if kw in text_lower]
    return ", ".join(matches) if matches else "other"

# -----------------------------------------------------------------------------
# Unit Tests (Assertions)
# -----------------------------------------------------------------------------
assert tag_commodities("Sugar delivery scheduled") == "sugar", \
    "[ASSERTION FAILED] 'Sugar delivery scheduled' must tag as 'sugar'"
assert tag_commodities("Unrelated office memo") == "other", \
    "[ASSERTION FAILED] 'Unrelated office memo' must tag as 'other'"
assert tag_commodities("Wheat harvest and packaging materials affected") == "wheat, packaging", \
    "[ASSERTION FAILED] Multi-keyword matching failed"

print("[UNIT TEST PASSED] Commodity keyword tagging assertions validated successfully.")

# Apply keyword tagging to cleaned datasets
df_emails_clean["tagged_keyword"] = df_emails_clean["text"].apply(tag_commodities)
df_news_clean["tagged_keyword"] = df_news_clean["text"].apply(tag_commodities)

print("[FEATURE ENGINEERING] Created 'tagged_keyword' column across all records.")
print("Distribution of commodity tags in combined corpus:")
print(pd.concat([df_emails_clean["tagged_keyword"], df_news_clean["tagged_keyword"]]).value_counts())

[UNIT TEST PASSED] Commodity keyword tagging assertions validated successfully.
[FEATURE ENGINEERING] Created 'tagged_keyword' column across all records.
Distribution of commodity tags in combined corpus:
tagged_keyword
wheat                14
sugar                13
packaging            11
cereal                9
cereal, packaging     1
Name: count, dtype: int64


In [ ]:
# =============================================================================
# Cell 5: FinBERT Sentiment Inference
# =============================================================================
# Combine emails and news into a unified analytical DataFrame
df_combined = pd.concat([df_emails_clean, df_news_clean], ignore_index=True)

# Detect GPU device for high-performance pipeline execution
device_idx = 0 if torch.cuda.is_available() else -1

print("[MODEL INITIALIZATION] Loading ProsusAI/finbert sentiment pipeline...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    truncation=True,
    max_length=512,
    device=device_idx
)

# Execute batched inference over the text column
BATCH_SIZE = 16
text_corpus = df_combined["text"].tolist()
print(f"[INFERENCE] Scoring {len(text_corpus)} texts using FinBERT in batches of {BATCH_SIZE}...")

raw_predictions = sentiment_pipeline(text_corpus, batch_size=BATCH_SIZE)

# Standardize labels to uppercase (POSITIVE, NEGATIVE, NEUTRAL) and round score to 4 decimal places
df_combined["sentiment_label"] = [res["label"].upper() for res in raw_predictions]
df_combined["sentiment_score"] = [round(float(res["score"]), 4) for res in raw_predictions]

print("[INFERENCE COMPLETE] Predictions and confidence probabilities successfully mapped.")
display(df_combined[["id", "source_type", "tagged_keyword", "text", "true_label", "sentiment_label", "sentiment_score"]].head())

[MODEL INITIALIZATION] Loading ProsusAI/finbert sentiment pipeline...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[INFERENCE] Scoring 48 texts using FinBERT in batches of 16...
[INFERENCE COMPLETE] Predictions and confidence probabilities successfully mapped.


,id,source_type,tagged_keyword,text,true_label,sentiment_label,sentiment_score
0,1,supplier_email,wheat,Invoice #402 for wheat delivery processed succ...,NEUTRAL,POSITIVE,0.8539
1,2,supplier_email,wheat,Bumper harvest for wheat means early delivery ...,POSITIVE,NEGATIVE,0.7273
2,3,supplier_email,wheat,"Severe drought hitting wheat farms in Midwest,...",NEGATIVE,NEGATIVE,0.9466
3,4,supplier_email,cereal,Routine delivery scheduling for cereal shipmen...,NEUTRAL,NEUTRAL,0.9229
4,5,supplier_email,wheat,Severe drought continues across agricultural b...,NEGATIVE,NEGATIVE,0.9647


In [ ]:
# =============================================================================
# Cell 6: Business Logic — 2-Day Rolling Risk Escalation
# =============================================================================
# Convert date to datetime and sort chronologically
df_combined["date"] = pd.to_datetime(df_combined["date"])
df_combined.sort_values(by=["date", "id"], inplace=True)
df_combined.reset_index(drop=True, inplace=True)

# -----------------------------------------------------------------------------
# Step 6.1: Calculate Daily Net Risk per Commodity (Count of NEGATIVE sentiment)
# -----------------------------------------------------------------------------
# Unnest/explode multi-commodity tags for precise daily risk tracking
risk_rows = []
for idx, row in df_combined.iterrows():
    commodities = [c.strip() for c in row["tagged_keyword"].split(",")]
    for comm in commodities:
        if comm in COMMODITY_KEYWORDS:
            risk_rows.append({
                "row_idx": idx,
                "date": row["date"],
                "commodity": comm,
                "is_negative": 1 if row["sentiment_label"] == "NEGATIVE" else 0
            })

df_commodity_events = pd.DataFrame(risk_rows)

# Compute daily net risk count (total negative sentiment events per commodity per date)
daily_net_risk = (
    df_commodity_events.groupby(["date", "commodity"])["is_negative"]
    .sum()
    .reset_index()
    .rename(columns={"is_negative": "negative_count"})
)

print("[BUSINESS LOGIC] Sample Daily Net Risk Counts (Negative Sentiment):")
display(daily_net_risk[daily_net_risk["negative_count"] > 0].head(8))

# -----------------------------------------------------------------------------
# Step 6.2: Compute 2-Day Consecutive Rolling Escalation Matrix
# Rule:
# - If negative sentiment occurs on consecutive days for a commodity -> 'High'
# - If negative sentiment occurs on a single day -> 'Watch'
# - Otherwise -> 'Normal'
# -----------------------------------------------------------------------------
# Map each commodity's calendar dates with negative sentiment
commodity_negative_dates = {
    comm: set(daily_net_risk[(daily_net_risk["commodity"] == comm) & (daily_net_risk["negative_count"] > 0)]["date"])
    for comm in COMMODITY_KEYWORDS
}

def determine_escalation(row: pd.Series) -> str:
    if row["sentiment_label"] != "NEGATIVE":
        return "Normal"

    current_date = row["date"]
    commodities = [c.strip() for c in row["tagged_keyword"].split(",")]

    is_high = False
    for comm in commodities:
        if comm in COMMODITY_KEYWORDS:
            # Check if previous calendar day (current_date - 1 day) had negative sentiment
            yesterday = current_date - pd.Timedelta(days=1)
            if yesterday in commodity_negative_dates.get(comm, set()):
                is_high = True
                break

    if is_high:
        return "High"
    return "Watch"

df_combined["escalation_level"] = df_combined.apply(determine_escalation, axis=1)

# Print High Risk alerts to console
print("\n" + "!"*70)
print("[CRITICAL ALERT] 2-DAY ROLLING HIGH RISK ESCALATIONS IDENTIFIED:")
print("!"*70)
high_risk_records = df_combined[df_combined["escalation_level"] == "High"]
if not high_risk_records.empty:
    display(high_risk_records[["date", "source_type", "tagged_keyword", "text", "sentiment_label", "escalation_level"]])
else:
    print("No records escalated to High risk level.")

[BUSINESS LOGIC] Sample Daily Net Risk Counts (Negative Sentiment):


,date,commodity,negative_count
0,2023-10-01,wheat,1
1,2023-10-02,sugar,1
2,2023-10-02,wheat,1
4,2023-10-03,wheat,1
6,2023-10-04,wheat,1
7,2023-10-05,sugar,1
8,2023-10-05,wheat,1
10,2023-10-06,wheat,1



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
[CRITICAL ALERT] 2-DAY ROLLING HIGH RISK ESCALATIONS IDENTIFIED:
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


,date,source_type,tagged_keyword,text,sentiment_label,escalation_level
2,2023-10-02,supplier_email,wheat,Bumper harvest for wheat means early delivery ...,NEGATIVE,High
4,2023-10-03,supplier_email,wheat,"Severe drought hitting wheat farms in Midwest,...",NEGATIVE,High
7,2023-10-04,supplier_email,wheat,Severe drought continues across agricultural b...,NEGATIVE,High
11,2023-10-05,news_article,wheat,Severe drought conditions in key wheat-produci...,NEGATIVE,High
13,2023-10-06,news_article,wheat,"Severe drought worsens across grain belt, whea...",NEGATIVE,High
18,2023-10-08,news_article,packaging,Surging pulp packaging costs place severe stra...,NEGATIVE,High
22,2023-10-11,news_article,sugar,Nationwide dock strikes threaten to paralyze k...,NEGATIVE,High
23,2023-10-12,supplier_email,sugar,"Port strikes continue unabated, sugar raw deli...",NEGATIVE,High
24,2023-10-12,news_article,sugar,Prolonged port strikes shut down maritime frei...,NEGATIVE,High


In [ ]:
# =============================================================================
# Cell 7: Model Evaluation, Error Analysis & Metrics
# =============================================================================
# Filter out NEUTRAL ground-truth records for binary evaluation on POSITIVE vs NEGATIVE
df_eval = df_combined[df_combined["true_label"].isin(["POSITIVE", "NEGATIVE"])].copy()

y_true = df_eval["true_label"]
y_pred = df_eval["sentiment_label"]

# Compute standalone core classification metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("="*60)
print("MODEL EVALUATION METRICS (BINARY POSITIVE vs NEGATIVE)")
print("="*60)
print(f"Accuracy:        {acc:.4f}")
print(f"Precision (W):   {prec:.4f}")
print(f"Recall (W):      {rec:.4f}")
print(f"F1-Score (W):    {f1:.4f}")

print("\n--- Scikit-Learn Classification Report ---")
print(classification_report(y_true, y_pred, zero_division=0))

print("--- Confusion Matrix ---")
labels_order = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
cm = confusion_matrix(y_true, y_pred, labels=labels_order)
cm_df = pd.DataFrame(cm, index=[f"True_{l}" for l in labels_order], columns=[f"Pred_{l}" for l in labels_order])
display(cm_df)

# -----------------------------------------------------------------------------
# Confidence Score Distribution: Correct vs Incorrect Predictions
# -----------------------------------------------------------------------------
df_eval["is_correct"] = (df_eval["true_label"] == df_eval["sentiment_label"])
confidence_stats = (
    df_eval.groupby("is_correct")["sentiment_score"]
    .agg(mean="mean", min="min", max="max", count="count")
    .reset_index()
)
print("\n--- Confidence Score Distribution (Correct vs Incorrect) ---")
display(confidence_stats)

# -----------------------------------------------------------------------------
# Error Analysis & Misclassified Records Table
# -----------------------------------------------------------------------------
misclassified_df = df_eval[~df_eval["is_correct"]]
print("\n" + "="*60)
print(f"MISCLASSIFIED ERROR ANALYSIS ({len(misclassified_df)} rows)")
print("="*60)
if not misclassified_df.empty:
    display(misclassified_df[["id", "source_type", "tagged_keyword", "text", "true_label", "sentiment_label", "sentiment_score"]])
else:
    print("Zero misclassifications detected across the evaluated test split.")

MODEL EVALUATION METRICS (BINARY POSITIVE vs NEGATIVE)
Accuracy:        0.8108
Precision (W):   0.8707
Recall (W):      0.8108
F1-Score (W):    0.8287

--- Scikit-Learn Classification Report ---
              precision    recall  f1-score   support

    NEGATIVE       0.81      0.94      0.87        18
     NEUTRAL       0.00      0.00      0.00         0
    POSITIVE       0.93      0.68      0.79        19

    accuracy                           0.81        37
   macro avg       0.58      0.54      0.55        37
weighted avg       0.87      0.81      0.83        37

--- Confusion Matrix ---


,Pred_NEGATIVE,Pred_NEUTRAL,Pred_POSITIVE
True_NEGATIVE,17,0,1
True_NEUTRAL,0,0,0
True_POSITIVE,4,2,13



--- Confidence Score Distribution (Correct vs Incorrect) ---


,is_correct,mean,min,max,count
0,False,0.818386,0.5829,0.9732,7
1,True,0.913083,0.6144,0.9723,30



MISCLASSIFIED ERROR ANALYSIS (7 rows)


,id,source_type,tagged_keyword,text,true_label,sentiment_label,sentiment_score
1,101,news_article,wheat,Global wheat market commodity prices drop shar...,POSITIVE,NEGATIVE,0.9732
2,2,supplier_email,wheat,Bumper harvest for wheat means early delivery ...,POSITIVE,NEGATIVE,0.7273
15,107,news_article,sugar,Sugar refineries invest in on-site solar energ...,POSITIVE,NEUTRAL,0.8731
20,110,news_article,cereal,Top cereal manufacturers announce consumer pri...,POSITIVE,NEGATIVE,0.9238
25,17,supplier_email,cereal,New organic cereal recipe approved for commerc...,POSITIVE,NEUTRAL,0.5829
33,117,news_article,sugar,Sugar spot prices surge 12% following catastro...,NEGATIVE,POSITIVE,0.7799
37,119,news_article,wheat,Agricultural authorities confirm crop disease ...,POSITIVE,NEGATIVE,0.8685


In [ ]:
# =============================================================================
# Cell 8: Final Export & Execution Summary Log
# =============================================================================
# Reorder columns precisely according to production specification
EXPORT_COLUMNS = [
    "id",
    "date",
    "source_type",
    "tagged_keyword",
    "text",
    "true_label",
    "sentiment_label",
    "sentiment_score",
    "escalation_level"
]

df_export = df_combined[EXPORT_COLUMNS].copy()
# Format date to standard string YYYY-MM-DD
df_export["date"] = df_export["date"].dt.strftime("%Y-%m-%d")

output_csv_path = os.path.join(PROCESSED_DATA_DIR, "sentiment_output.csv")
df_export.to_csv(output_csv_path, index=False, encoding="utf-8")

# Execution Summary Audit Log
total_ingested = emails_init + news_init
total_dropped = emails_dropped + news_dropped
total_processed = len(df_export)
breakdown = df_export["escalation_level"].value_counts().to_dict()

print("="*70)
print("           RETAIL SUPPLY CHAIN NLP PIPELINE - EXECUTION AUDIT")
print("="*70)
print(f"Total Ingested Records (Raw):     {total_ingested}")
print(f"Total Cleaned / Dropped Records:  {total_dropped}")
print(f"Total Clean Analytical Records:   {total_processed}")
print("-"*70)
print("Escalation Risk Level Breakdown:")
for level in ["High", "Watch", "Normal"]:
    count = breakdown.get(level, 0)
    pct = (count / total_processed) * 100
    print(f"  * {level:<8}: {count:2d} ({pct:5.1f}%)")
print("-"*70)
print(f"SUCCESS: Production dataset exported to: {output_csv_path}")
print("Status: Fully validated, production-style, and ready for Power BI consumption.")
print("="*70)

           RETAIL SUPPLY CHAIN NLP PIPELINE - EXECUTION AUDIT
Total Ingested Records (Raw):     52
Total Cleaned / Dropped Records:  4
Total Clean Analytical Records:   48
----------------------------------------------------------------------
Escalation Risk Level Breakdown:
  * High    :  9 ( 18.8%)
  * Watch   : 14 ( 29.2%)
  * Normal  : 25 ( 52.1%)
----------------------------------------------------------------------
SUCCESS: Production dataset exported to: ./data/processed/sentiment_output.csv
Status: Fully validated, production-style, and ready for Power BI consumption.
